<a href="https://colab.research.google.com/github/HatolkarAV/ICU-Weaning-Prediction/blob/main/notebooks/Data_PreProcessing/07_Rolling_Origin_Windowing_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Objective

*   Reload the cleaned full-history long table and re-attach imv_end, restoring the pre-48h data that Notebook 06's fixed window discarded.
*   Keep only visits with enough pre-extubation history to support at least one valid origin, and count how many origin each visit can supply.
*   Generate rolling origins t at a fixed stride through the pre-imv_end period, so one visit yields many windows instead of one.
*   Cut one sample per origin: 48hr of context behind t, targets at +1hr/+4hr/+8hr ahead of t, with every window ending strictly before imv_end.
*   Split by the patient(GroupShuffleSplit on visit_occurence+id) before any fitting, so no visit's windows leak across train/val/test.
*   Apply stage-2 median fill and scaling with satistics fitted on the training windows only.
*   Save two windowed artifacts - a filled grid (Tier1/2) and raw triplets(Tier 3) - plus updated metadata.










### Google Drive setup and project folders

Mount Drive and point to the project folders. This notebook reads the two tables
saved by notebook 06 and writes its three window files back to Drive.

In [1]:
# Project setup - keep this cell identical in every notebook
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/ventilator_weaning')

COHORT_DIR     = PROJECT_DIR / 'cohort'
PREPROCESS_DIR = PROJECT_DIR / 'preprocessing'
WINDOWS_DIR    = PROJECT_DIR / 'windows'
RESULTS_DIR    = PROJECT_DIR / 'results'
FIGURES_DIR    = PROJECT_DIR / 'figures'

# Safe to run again, does nothing if the folders already exist
for d in [COHORT_DIR, PREPROCESS_DIR, WINDOWS_DIR, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Files this notebook reads (written by notebook 06)
CLEAN_LONG_PATH = PREPROCESS_DIR / 'cohort_features_long_clean.parquet'
ANCHORED_PATH   = PREPROCESS_DIR / 'cohort_features_anchored.parquet'
FILLED_PATH     = PREPROCESS_DIR / 'cohort_wide_filled.parquet'

# Files this notebook writes
GRID_PATH     = WINDOWS_DIR / 'windows_tier12_grid.npz'
TRIPLETS_PATH = WINDOWS_DIR / 'windows_tier3_triplets.parquet'
METADATA_PATH = WINDOWS_DIR / 'windows_metadata.json'

# Stop early with a clear message if an input file is missing
for p in [CLEAN_LONG_PATH, ANCHORED_PATH, FILLED_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Cannot find {p}. Run notebook 06 first.")

print('Project folder:', PROJECT_DIR)
print('All input files found, ready to start.')

Mounted at /content/drive
Project folder: /content/drive/MyDrive/ventilator_weaning
All input files found, ready to start.


Drive is mounted and the folders exist. The check stops the notebook now if a
notebook 06 output is missing, instead of failing later.

### Imports and configuration

**Initial BigQuery setup by Ayushi Kashyap - adapted for this notebook.**

In [2]:
import pandas as pd
import numpy as np
from google.cloud import bigquery
import matplotlib.pyplot as plt
import json

In [3]:
PROJECT_ID         = 'capstoneweaningprediction' #@param {type:"string"}
DATASET_PROJECT_ID = 'amsterdamumcdb'            #@param {type:"string"}
DATASET_ID         = 'version1_5_0'              #@param {type:"string"}
LOCATION           = 'eu'                        #@param {type:"string"}

In [4]:
import os
from google.colab import auth
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
auth.authenticate_user()
print('Authenticated')

Authenticated


In [5]:
%load_ext google.colab.data_table
from google.colab.data_table import DataTable
DataTable.max_columns = 50
DataTable.max_rows    = 80000

In [6]:
%load_ext bigquery_magics
from bigquery_magics import bigquery_magics
def_config = bigquery.job.QueryJobConfig(
    default_dataset=DATASET_PROJECT_ID + '.' + DATASET_ID
)
bigquery_magics.context.default_query_job_config = def_config
client = bigquery.Client(
    project=PROJECT_ID, location=LOCATION,
    default_query_job_config=def_config
)
print('BigQuery client ready')

BigQuery client ready


### 1. Load cleaned long table and re-attach the anchor

Load the cleaned full-history measurement table and re-attach the imv_end anchor, so every reading carries its time position relative to extubation. This restores the pre-48h data that Notebook 06's fixed window discarded.

In [7]:
# Window settings
CONTEXT_HOURS = 48       # Hours used as input
T_RANGE_HOURS = 24       # Origins start this many hours before IMV end
T_STRIDE_HOURS = 4       # Gap between consecutive origins
HORIZONS = [1, 4, 8]     # Forecast horizons

# Load the cleaned long-format data from Drive
long_clean = pd.read_parquet(CLEAN_LONG_PATH)

# Get one IMV end time for each visit
anchored = pd.read_parquet(ANCHORED_PATH)
imv_end_map = anchored[
    ["visit_occurrence_id", "imv_end"]
].drop_duplicates()

# Add IMV end time to the cleaned data
df = long_clean.merge(
    imv_end_map,
    on="visit_occurrence_id",
    how="inner"
)

# Convert timestamps
df["measurement_datetime"] = pd.to_datetime(
    df["measurement_datetime"],
    utc=True
)
df["imv_end"] = pd.to_datetime(
    df["imv_end"],
    utc=True
)

# Calculate time relative to IMV end
time_delta = df["measurement_datetime"] - df["imv_end"]
df["hours_since_imv_end"] = time_delta.dt.total_seconds() / 3600

print(f"Visits in anchored file:  {len(imv_end_map):,}")
print(f"Rows loaded:              {len(df):,}")
print(f"Visits with anchor:       {df['visit_occurrence_id'].nunique():,}")
print(f"Hours range:              {df['hours_since_imv_end'].min():.0f} to {df['hours_since_imv_end'].max():.0f}")
print(f"Rows before IMV end:      {(df['hours_since_imv_end'] < 0).sum():,}")
print(f"Features available:       {df['feature_name'].nunique()}")

Visits in anchored file:  1,997
Rows loaded:              4,849,902
Visits with anchor:       1,997
Hours range:              -3116 to 4705
Rows before IMV end:      3,592,557
Features available:       13


###2. Find valid prediction windows

In [8]:
# Possible prediction times based on the selected stride
raw_candidates = list(range(-T_RANGE_HOURS, 0, T_STRIDE_HOURS))

# Remove prediction times where the target would go past IMV end
max_horizon = max(HORIZONS)
valid_origins = []

for t in raw_candidates:
    if t + max_horizon <= 0:
        valid_origins.append(t)

print(f"Raw prediction times:   {raw_candidates}")
print(f"Valid prediction times: {valid_origins}")
print(f"Dropped:                {sorted(set(raw_candidates) - set(valid_origins))}")
print(f"Maximum windows/visit:  {len(valid_origins)}\n")

# Earliest hour available for each visit
earliest_hour = df.groupby("visit_occurrence_id")["hours_since_imv_end"].min()

# Count how many windows can be created for each visit
def count_usable_origins(earliest):
    usable = 0

    for t in valid_origins:
        if (t - CONTEXT_HOURS) >= earliest:
            usable += 1

    return usable

origins_per_visit = earliest_hour.apply(count_usable_origins)

eligible_visits = origins_per_visit[origins_per_visit > 0]
total_windows = origins_per_visit.sum()

print("Windows available per visit:")
print(origins_per_visit.value_counts().sort_index().to_string())

# Read the fixed-window cohort size from notebook 06, do not hardcode it
fixed_window_visits = pd.read_parquet(
    FILLED_PATH,
    columns=["visit_occurrence_id"]
)["visit_occurrence_id"].nunique()

print(f"\nEligible visits:                {len(eligible_visits):,}")
print(f"Total windows:                  {total_windows:,}")
print(f"Fixed-window baseline visits:   {fixed_window_visits:,}")
print(f"Increase in samples:            {total_windows / fixed_window_visits:.1f}x")

Raw prediction times:   [-24, -20, -16, -12, -8, -4]
Valid prediction times: [-24, -20, -16, -12, -8]
Dropped:                [-4]
Maximum windows/visit:  5

Windows available per visit:
hours_since_imv_end
0     377
1      24
2      35
3      39
4      40
5    1482

Eligible visits:                1,620
Total windows:                  7,781
Fixed-window baseline visits:   1,980
Increase in samples:            3.9x


Most visits produced multiple prediction windows, resulting in 8,202 samples from 1,715 eligible visits.

### 3. Build the prediction window index

Create one row for each valid prediction window, storing the visit ID, prediction time, context start and end, and the target time points (+1h, +4h, and +8h).
Claude Opus 4.8: Build forecasting windows per (visit, origin) and validate context/target integrity.

In [9]:
window_rows = []

for visit_id, earliest in earliest_hour.items():
    for t in valid_origins:

        context_start = t - CONTEXT_HOURS

        # Make sure enough history is available
        if context_start >= earliest:

            window_rows.append({
                "visit_occurrence_id": visit_id,
                "origin_t": t,
                "context_start": context_start,
                "context_end": t,
                "target_h1": t + HORIZONS[0],
                "target_h4": t + HORIZONS[1],
                "target_h8": t + HORIZONS[2]
            })

windows = pd.DataFrame(window_rows)

# Create a unique ID for each window
windows["window_id"] = (
    windows["visit_occurrence_id"].astype(str)
    + "_t"
    + windows["origin_t"].astype(str)
)

# Quick checks
targets_before_end = (
    windows[["target_h1", "target_h4", "target_h8"]] <= 0
).all().all()

correct_context = (
    (windows["context_end"] - windows["context_start"]) == CONTEXT_HOURS
).all()

print(f"Total windows: {len(windows):,}")
print(f"Visits: {windows['visit_occurrence_id'].nunique():,}")
print(f"Window IDs: {windows['window_id'].nunique():,}")

print("\nWindows at each prediction time:")
print(windows["origin_t"].value_counts().sort_index())

print(f"\nTargets before IMV end: {targets_before_end}")
print(f"Correct context length: {correct_context}")

print("\nFirst few windows:")
print(windows.head())

Total windows: 7,781
Visits: 1,620
Window IDs: 7,781

Windows at each prediction time:
origin_t
-24    1482
-20    1522
-16    1561
-12    1596
-8     1620
Name: count, dtype: int64

Targets before IMV end: True
Correct context length: True

First few windows:
   visit_occurrence_id  origin_t  context_start  context_end  target_h1  \
0                    5       -24            -72          -24        -23   
1                    5       -20            -68          -20        -19   
2                    5       -16            -64          -16        -15   
3                    5       -12            -60          -12        -11   
4                    5        -8            -56           -8         -7   

   target_h4  target_h8 window_id  
0        -20        -16    5_t-24  
1        -16        -12    5_t-20  
2        -12         -8    5_t-16  
3         -8         -4    5_t-12  
4         -4          0     5_t-8  


### 4. Build the master hourly grid and observation mask

Create one hourly grid for each eligible visit and add an observation mask to identify measured and missing values.

In [10]:
import numpy as np

# Define the hourly range
GRID_START = min(valid_origins) - CONTEXT_HOURS
GRID_END = max(valid_origins) + max(HORIZONS)

FEATURES = sorted(df["feature_name"].unique())
eligible_visits = sorted(windows["visit_occurrence_id"].unique())

print(f"Grid range: {GRID_START}h to {GRID_END}h ({GRID_END - GRID_START + 1} hours)")
print(f"Visits: {len(eligible_visits):,}")
print(f"Features: {len(FEATURES)}")

# Keep only eligible visits and the required time range
subset = df[df["visit_occurrence_id"].isin(eligible_visits)].copy()

subset = subset[
    (subset["hours_since_imv_end"] >= GRID_START) &
    (subset["hours_since_imv_end"] <= GRID_END)
]

# Convert timestamps to hourly bins
subset["hour_bin"] = np.floor(subset["hours_since_imv_end"]).astype(int)

# Average multiple values in the same hour
observed = (
    subset.groupby(
        ["visit_occurrence_id", "feature_name", "hour_bin"],
        as_index=False
    )["value_as_number"]
    .mean()
)

# Create the complete hourly grid
full_index = pd.MultiIndex.from_product(
    [eligible_visits, FEATURES, range(GRID_START, GRID_END + 1)],
    names=["visit_occurrence_id", "feature_name", "hour_bin"],
)

master_grid = (
    observed
    .set_index(["visit_occurrence_id", "feature_name", "hour_bin"])
    .reindex(full_index)
    .reset_index()
)

# Mark observed values
master_grid["is_observed"] = (
    master_grid["value_as_number"]
    .notna()
    .astype(int)
)

observed_rate = (
    master_grid.groupby("feature_name")["is_observed"].mean() * 100
)

print(f"\nMaster grid rows: {len(master_grid):,}")

print(
    f"Observed cells: {master_grid['is_observed'].sum():,} "
    f"({100 * master_grid['is_observed'].mean():.1f}%)"
)

print("\nObserved rate by feature (%):")
print(observed_rate.round(1).sort_values(ascending=False).to_string())

Grid range: -72h to 0h (73 hours)
Visits: 1,620
Features: 13

Master grid rows: 1,537,380
Observed cells: 983,549 (64.0%)

Observed rate by feature (%):
feature_name
heart_rate          89.6
diastolic_bp        88.5
mean_bp             88.5
systolic_bp         88.5
spo2                85.5
peep                75.8
respiratory_rate    75.8
fio2                75.0
tidal_volume        74.6
paco2               26.4
ph                  26.3
pao2                26.2
lactate             10.9


### 5. Fill gaps with LOCF (1st stage)

In [11]:
# Maximum forward-fill time (hours) for each feature
LOCF_CUTOFF_HOURS = {
    # Vitals
    "heart_rate": 4,
    "systolic_bp": 4,
    "diastolic_bp": 4,
    "mean_bp": 4,
    "spo2": 4,
    "respiratory_rate": 4,

    # Ventilator settings
    "fio2": 12,
    "peep": 12,
    "tidal_volume": 12,

    # Lab measurements
    "pao2": 12,
    "paco2": 12,
    "ph": 12,
    "lactate": 12,
}

# Sort the data
grid = master_grid.sort_values(
    ["visit_occurrence_id", "feature_name", "hour_bin"]
).copy()

groups = grid.groupby(
    ["visit_occurrence_id", "feature_name"],
    sort=False
)

# Forward fill
grid["value_filled"] = groups["value_as_number"].ffill()

# Save the last observed hour
grid["last_obs_hour"] = grid["hour_bin"].where(
    grid["is_observed"] == 1
)
grid["last_obs_hour"] = groups["last_obs_hour"].ffill()

# Time since the last observation
grid["gap_hours"] = (
    grid["hour_bin"] - grid["last_obs_hour"]
)

# Remove values that are too old
grid["cutoff"] = grid["feature_name"].map(
    LOCF_CUTOFF_HOURS
)

too_old = grid["gap_hours"] > grid["cutoff"]
grid.loc[too_old, "value_filled"] = np.nan

# Summary
filled_by_locf = (
    (grid["is_observed"] == 0)
    & grid["value_filled"].notna()
)

still_missing = grid["value_filled"].isna()

print(
    f"Observed cells: {grid['is_observed'].sum():,} "
    f"({100 * grid['is_observed'].mean():.1f}%)"
)

print(
    f"Filled by LOCF: {filled_by_locf.sum():,} "
    f"({100 * filled_by_locf.mean():.1f}%)"
)

print(
    f"Still missing: {still_missing.sum():,} "
    f"({100 * still_missing.mean():.1f}%)"
)

print(
    f"Coverage after: "
    f"{100 * grid['value_filled'].notna().mean():.1f}%"
)

coverage = grid.groupby("feature_name")["value_filled"].apply(
    lambda s: 100 * s.notna().mean()
)

print("\nCoverage after LOCF by feature (%):")
print(
    coverage.round(1)
    .sort_values(ascending=False)
    .to_string()
)

Observed cells: 983,549 (64.0%)
Filled by LOCF: 393,552 (25.6%)
Still missing: 160,279 (10.4%)
Coverage after: 89.6%

Coverage after LOCF by feature (%):
feature_name
heart_rate          95.9
spo2                95.4
diastolic_bp        94.9
systolic_bp         94.9
mean_bp             94.9
paco2               94.7
ph                  94.7
pao2                94.6
peep                91.0
tidal_volume        90.8
fio2                90.7
respiratory_rate    86.7
lactate             45.2


LOCF filled most short gaps, increasing overall data coverage to 89.7%, while lactate remained the least complete feature.

### 6. Split patients into train / validation / test

In [12]:
from sklearn.model_selection import GroupShuffleSplit

# Train, validation and test split
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15
RANDOM_STATE = 42

# One row per visit
visit_ids = windows["visit_occurrence_id"].unique()

# Split into train and holdout
gss = GroupShuffleSplit(
    n_splits=1,
    train_size=TRAIN_FRAC,
    random_state=RANDOM_STATE
)

train_pos, holdout_pos = next(
    gss.split(visit_ids, groups=visit_ids)
)

train_visits = visit_ids[train_pos]
holdout_visits = visit_ids[holdout_pos]

# Split holdout into validation and test
val_share = VAL_FRAC / (VAL_FRAC + TEST_FRAC)

gss = GroupShuffleSplit(
    n_splits=1,
    train_size=val_share,
    random_state=RANDOM_STATE
)

val_pos, test_pos = next(
    gss.split(holdout_visits, groups=holdout_visits)
)

val_visits = holdout_visits[val_pos]
test_visits = holdout_visits[test_pos]

train_set = set(train_visits)
val_set = set(val_visits)

def assign_split(visit_id):

    if visit_id in train_set:
        return "train"

    if visit_id in val_set:
        return "val"

    return "test"

windows["split"] = windows["visit_occurrence_id"].apply(assign_split)

# Check for overlap
overlap_train_val = len(set(train_visits) & set(val_visits))
overlap_train_test = len(set(train_visits) & set(test_visits))
overlap_val_test = len(set(val_visits) & set(test_visits))

print(f"Train visits: {len(train_visits):,}")
print(f"Validation visits: {len(val_visits):,}")
print(f"Test visits: {len(test_visits):,}")

print(f"\nTrain windows: {(windows['split'] == 'train').sum():,}")
print(f"Validation windows: {(windows['split'] == 'val').sum():,}")
print(f"Test windows: {(windows['split'] == 'test').sum():,}")

print(f"\nTrain/Val overlap: {overlap_train_val}")
print(f"Train/Test overlap: {overlap_train_test}")
print(f"Val/Test overlap: {overlap_val_test}")

Train visits: 1,134
Validation visits: 243
Test visits: 243

Train windows: 5,463
Validation windows: 1,169
Test windows: 1,149

Train/Val overlap: 0
Train/Test overlap: 0
Val/Test overlap: 0


### 7. Stage-2 median fill and scaling (fit on training visits only)

In [13]:
from sklearn.preprocessing import StandardScaler

# Add the train/val/test split to the grid
split_map = (
    windows[["visit_occurrence_id", "split"]]
    .drop_duplicates()
)

grid = grid.merge(
    split_map,
    on="visit_occurrence_id",
    how="left"
)

# Calculate the median from the training data
train_rows = grid[grid["split"] == "train"]

feature_medians = (
    train_rows
    .groupby("feature_name")["value_filled"]
    .median()
)

print("Training median for each feature:")
print(feature_medians.round(2).to_string())

# Fill the remaining missing values
grid["median_for_feature"] = grid["feature_name"].map(feature_medians)

grid["value_complete"] = grid["value_filled"].fillna(
    grid["median_for_feature"]
)

print(
    f"\nMissing values after median fill: "
    f"{grid['value_complete'].isna().sum()}"
)

# Scale each feature using the training data
scalers = {}

for feature in FEATURES:

    train_values = grid.loc[
        (grid["feature_name"] == feature) &
        (grid["split"] == "train"),
        "value_complete"
    ].values.reshape(-1, 1)

    scaler = StandardScaler()
    scaler.fit(train_values)

    scalers[feature] = scaler

grid["value_scaled"] = np.nan

for feature in FEATURES:

    mask = grid["feature_name"] == feature

    values = grid.loc[
        mask,
        "value_complete"
    ].values.reshape(-1, 1)

    grid.loc[mask, "value_scaled"] = (
        scalers[feature]
        .transform(values)
        .ravel()
    )

# Check the training data after scaling
train_scaled = grid[grid["split"] == "train"]

check = (
    train_scaled
    .groupby("feature_name")["value_scaled"]
    .agg(["mean", "std"])
)

print("\nTraining data after scaling:")
print(check.round(2).to_string())

Training median for each feature:
feature_name
diastolic_bp         61.00
fio2                 40.00
heart_rate           87.00
lactate               1.10
mean_bp              83.00
paco2                 5.60
pao2                 12.13
peep                  7.00
ph                    7.43
respiratory_rate     20.00
spo2                 97.00
systolic_bp         129.00
tidal_volume        457.00

Missing values after median fill: 0

Training data after scaling:
                  mean  std
feature_name               
diastolic_bp      -0.0  1.0
fio2               0.0  1.0
heart_rate        -0.0  1.0
lactate            0.0  1.0
mean_bp           -0.0  1.0
paco2             -0.0  1.0
pao2              -0.0  1.0
peep              -0.0  1.0
ph                 0.0  1.0
respiratory_rate  -0.0  1.0
spo2              -0.0  1.0
systolic_bp        0.0  1.0
tidal_volume       0.0  1.0


### 8. Cut context and target arrays for each window

In [14]:
# Create lookups
grid_indexed = (
    grid.set_index(
        ["visit_occurrence_id", "feature_name", "hour_bin"]
    )
    .sort_index()
)

scaled_lookup = grid_indexed["value_scaled"]
observed_lookup = grid_indexed["is_observed"]

# Context hours
context_offsets = list(range(-CONTEXT_HOURS, 0))

context_arrays = []
context_masks = []
target_arrays = []
target_masks = []
kept_window_ids = []

for row in windows.itertuples(index=False):

    visit_id = row.visit_occurrence_id
    origin = row.origin_t

    # Context
    context_hours = [origin + off for off in context_offsets]

    ctx_values = np.zeros((CONTEXT_HOURS, len(FEATURES)))
    ctx_observed = np.zeros((CONTEXT_HOURS, len(FEATURES)))

    for f_idx, feature in enumerate(FEATURES):
        for h_idx, hour in enumerate(context_hours):
            ctx_values[h_idx, f_idx] = scaled_lookup[(visit_id, feature, hour)]
            ctx_observed[h_idx, f_idx] = observed_lookup[(visit_id, feature, hour)]

    # Targets
    target_hours = [origin + h for h in HORIZONS]

    tgt_values = np.zeros((len(HORIZONS), len(FEATURES)))
    tgt_observed = np.zeros((len(HORIZONS), len(FEATURES)))

    for f_idx, feature in enumerate(FEATURES):
        for h_idx, hour in enumerate(target_hours):
            tgt_values[h_idx, f_idx] = scaled_lookup[(visit_id, feature, hour)]
            tgt_observed[h_idx, f_idx] = observed_lookup[(visit_id, feature, hour)]

    context_arrays.append(ctx_values)
    context_masks.append(ctx_observed)
    target_arrays.append(tgt_values)
    target_masks.append(tgt_observed)
    kept_window_ids.append(row.window_id)

# Convert to NumPy arrays
X_context = np.stack(context_arrays)
X_context_mask = np.stack(context_masks)
y_target = np.stack(target_arrays)
y_target_mask = np.stack(target_masks)

print(f"Context array shape: {X_context.shape}")
print(f"Context mask shape:  {X_context_mask.shape}")
print(f"Target array shape:  {y_target.shape}")
print(f"Target mask shape:   {y_target_mask.shape}")
print(f"Windows kept:        {len(kept_window_ids):,}")

target_observed_rate = 100 * y_target_mask.mean()

print(f"\nObserved target values: {target_observed_rate:.1f}%")

Context array shape: (7781, 48, 13)
Context mask shape:  (7781, 48, 13)
Target array shape:  (7781, 3, 13)
Target mask shape:   (7781, 3, 13)
Windows kept:        7,781

Observed target values: 64.7%


All windows are successfully converted into model-ready input and target arrays, with about 65% of the target values available for evaluation.

### 9. Save windowed artifacts and metadata

In [15]:
# Split and visit for each window
window_split_map = windows.set_index("window_id")["split"]
window_visit_map = windows.set_index("window_id")["visit_occurrence_id"]

split_per_window = np.array(
    [window_split_map[wid] for wid in kept_window_ids]
)

visit_per_window = np.array(
    [window_visit_map[wid] for wid in kept_window_ids]
)

# Save Tier 1 and Tier 2 data to Drive
np.savez_compressed(
    GRID_PATH,
    X_context=X_context,
    X_context_mask=X_context_mask,
    y_target=y_target,
    y_target_mask=y_target_mask,
    window_ids=np.array(kept_window_ids),
    visit_ids=visit_per_window,
    split=split_per_window,
)

print(f"Saved Tier 1/2 grid -> {GRID_PATH}")

# Save Tier 3 data
kept_visits = set(visit_per_window)

triplets = df[
    (df["visit_occurrence_id"].isin(kept_visits))
    & (df["hours_since_imv_end"] >= GRID_START)
    & (df["hours_since_imv_end"] < 0)
][
    [
        "visit_occurrence_id",
        "feature_name",
        "hours_since_imv_end",
        "value_as_number",
    ]
].copy()

triplets = triplets.merge(
    windows[["visit_occurrence_id", "split"]].drop_duplicates(),
    on="visit_occurrence_id",
    how="left",
)

triplets.to_parquet(TRIPLETS_PATH)

print(f"Saved Tier 3 triplets -> {TRIPLETS_PATH} ({len(triplets):,} rows)")

# Save metadata
scaler_stats = {
    feature: {
        "mean": float(scalers[feature].mean_[0]),
        "std": float(scalers[feature].scale_[0]),
        "median_fill": float(feature_medians[feature]),
    }
    for feature in FEATURES
}

metadata = {
    "context_hours": CONTEXT_HOURS,
    "t_range_hours": T_RANGE_HOURS,
    "t_stride_hours": T_STRIDE_HOURS,
    "horizons": HORIZONS,
    "valid_origins": valid_origins,
    "grid_start": int(GRID_START),
    "grid_end": int(GRID_END),
    "features": FEATURES,
    "n_windows": len(kept_window_ids),
    "n_visits": int(len(kept_visits)),
    "split_fractions": {
        "train": TRAIN_FRAC,
        "val": VAL_FRAC,
        "test": TEST_FRAC,
    },
    "split_counts": {
        "train_visits": int(len(train_visits)),
        "val_visits": int(len(val_visits)),
        "test_visits": int(len(test_visits)),
    },
    "units_note": "paco2 and pao2 are in kPa (AmsterdamUMCdb convention)",
    "scaler_stats": scaler_stats,
    "random_state": RANDOM_STATE,
}

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved metadata -> {METADATA_PATH}")

# Read all three files back so we know they really saved
check_grid = np.load(GRID_PATH, allow_pickle=True)
check_trip = pd.read_parquet(TRIPLETS_PATH)
with open(METADATA_PATH) as f:
    check_meta = json.load(f)

assert check_grid["X_context"].shape == X_context.shape, "Grid shape does not match."
assert len(check_trip) == len(triplets), "Triplet row count does not match."
assert check_meta["n_windows"] == len(kept_window_ids), "Metadata does not match."

print()
print("All three files confirmed on Drive:")
print(f"Tier 1/2 grid:   {X_context.shape[0]:,} windows, shape {X_context.shape}")
print(f"Tier 3 triplets: {len(triplets):,} rows")
print(f"Metadata:        {len(metadata)} fields")
print()
print(f"WINDOWS: {len(kept_window_ids):,} from {len(kept_visits):,} visits")
print(f"Split (visits):  train {len(train_visits):,} | val {len(val_visits):,} | test {len(test_visits):,}")

Saved Tier 1/2 grid -> /content/drive/MyDrive/ventilator_weaning/windows/windows_tier12_grid.npz
Saved Tier 3 triplets -> /content/drive/MyDrive/ventilator_weaning/windows/windows_tier3_triplets.parquet (981,702 rows)
Saved metadata -> /content/drive/MyDrive/ventilator_weaning/windows/windows_metadata.json

All three files confirmed on Drive:
Tier 1/2 grid:   7,781 windows, shape (7781, 48, 13)
Tier 3 triplets: 981,702 rows
Metadata:        15 fields

WINDOWS: 7,781 from 1,620 visits
Split (visits):  train 1,134 | val 243 | test 243
